### SETUP MODULES
The following modules include imports / IBM account initialization that the homework requires.

In [35]:
# -----------------------------------
# IMPORTS REQUIRED FOR THE ASSIGNMENT
# -----------------------------------
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from qiskit import QuantumCircuit
from qiskit.visualization import plot_histogram
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime import QiskitRuntimeService

In [36]:
# -------------------------------
# ACCOUNT LINK TO QISKIT SERVVICE
# -------------------------------
QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token="hPp27YYsQIgcnApUBSMsQnMIBL-VkUwIHN5-aOHeKnkF",
    instance="crn:v1:bluemix:public:quantum-computing:us-east:a/333c753dd333451a9f49f7993cede44f:95cf404b-9d0a-49c0-9690-bc0a1b6fc509::",
    overwrite=True,
    set_as_default=True
)

service = QiskitRuntimeService()
print("Connected to IBM Quantum.")

Connected to IBM Quantum.


### PART 1
In this part, I have:

- Chosen 2 available backends
- Read / store all properties from each qubit
- Find the min, max, median and mean for each property

In [37]:
# -------------------------------
# INITIALZING / CHOOSING BACKENDS
# -------------------------------
available_backends = service.backends(
    simulator=False,
    operational=True
)

# Viewing all potential backends
for backend in available_backends:
    print(backend)

# Selecting backend here
backend_one = service.backend('ibm_fez')
backend_two = service.backend('ibm_marrakesh')

<IBMBackend('ibm_fez')>
<IBMBackend('ibm_marrakesh')>
<IBMBackend('ibm_kingston')>


In [65]:
# ---------------------------------
# READING BACKEND CHARACTERIZATIONS
# ---------------------------------
device_info = {
    "ibm_fez": {"T1": [], "T2": [], "Readout": []},
    "ibm_marrakesh": {"T1": [], "T2": [], "Readout": []},
}

# The number of qubits for both backends
num_qubits_b1 = backend_one.num_qubits
num_qubits_b2 = backend_two.num_qubits

# Preparing the properties for each backend to be read from
backend_one_prop = backend_one.properties(refresh=True)
backend_two_prop = backend_two.properties(refresh=True)

# Reading error from IBM_FEZ
for qubit in range(num_qubits_b1):
    qubit_properties = backend_one_prop.qubit_property(qubit)
    if "T1" in qubit_properties:
        device_info["ibm_fez"]["T1"].append(qubit_properties["T1"][0] * 1e6)
    else:
        device_info["ibm_fez"]["T1"].append(np.nan)

    if "T2" in qubit_properties:
        device_info["ibm_fez"]["T2"].append(qubit_properties["T2"][0] * 1e6)
    else:
        device_info["ibm_fez"]["T2"].append(np.nan)

    if "readout_error" in qubit_properties:
        device_info["ibm_fez"]["Readout"].append(qubit_properties["readout_error"][0] * 100)
    else:
        device_info["ibm_fez"]["Readout"].append(np.nan)

# Reading error from IBM_MERRAKESH
for qubit in range(num_qubits_b2):
    qubit_properties = backend_two_prop.qubit_property(qubit)

    if "T1" in qubit_properties:
        device_info["ibm_marrakesh"]["T1"].append(qubit_properties["T1"][0] * 1e6)
    else:
        device_info["ibm_marrakesh"]["T1"].append(np.nan)

    if "T2" in qubit_properties:
        device_info["ibm_marrakesh"]["T2"].append(qubit_properties["T2"][0] * 1e6)
    else:
        device_info["ibm_marrakesh"]["T2"].append(np.nan)

    if "readout_error" in qubit_properties:
        device_info["ibm_marrakesh"]["Readout"].append(qubit_properties["readout_error"][0] * 100)
    else:
        device_info["ibm_marrakesh"]["Readout"].append(np.nan)
   
# fez_df = pd.DataFrame(device_info["ibm_fez"])
# merrakesh_df = pd.DataFrame(device_info["ibm_marrakesh"])
# display(fez_df.style.set_caption("IBM FEZ"))
# display(merrakesh_df.style.set_caption("IBM MARRAKESH"))

In [66]:
# ---------------------------------
# READING GATE ERRORS
# ---------------------------------
target_fez = backend_one.target
target_marrakesh = backend_two.target

# Adding gate error field to each device
device_info["ibm_fez"]["SingleQubitGateErrors"] = {}
device_info["ibm_fez"]["TwoQubitGateErrors"] = {}
device_info["ibm_marrakesh"]["SingleQubitGateErrors"] = {}
device_info["ibm_marrakesh"]["TwoQubitGateErrors"] = {}

# Excluding non-computational/global gates
excluded_instructions = {
    "measure",
    "measure_reset",
    "measure_reset_2",
    "reset",
    "reset_2",
    "delay",
    "barrier"
}

# Reading gate errors from IBM_FEZ
for gate_name in sorted(target_fez.operation_names):
    if gate_name in excluded_instructions:
        continue

    print(gate_name)


    qubit_arguments = target_fez.qargs_for_operation_name(gate_name)
    if qubit_arguments is None:
        continue

    for qubits_used in qubit_arguments:
        gate_properties = target_fez[gate_name][qubits_used]
        if gate_properties is None or gate_properties.error is None:
            error = np.nan
        else:
            error = gate_properties.error * 100

        if len(qubits_used) == 1:
            device_info["ibm_fez"]["SingleQubitGateErrors"].setdefault(gate_name, []).append(error)
        elif len(qubits_used) == 2:
            device_info["ibm_fez"]["TwoQubitGateErrors"].setdefault(gate_name, []).append(error)

# Reading gate errors from IBM_MARRAKESH
for gate_name in sorted(target_marrakesh.operation_names):
    if gate_name in excluded_instructions:
        continue

    qubit_arguments = target_marrakesh.qargs_for_operation_name(gate_name)
    if qubit_arguments is None:
        continue

    for qubits_used in qubit_arguments:
        gate_properties = target_marrakesh[gate_name][qubits_used]
        if gate_properties is None or gate_properties.error is None:
            error = np.nan
        else:
            error = gate_properties.error * 100

        if len(qubits_used) == 1:
            device_info["ibm_marrakesh"]["SingleQubitGateErrors"].setdefault(gate_name, []).append(error)
        elif len(qubits_used) == 2:
            device_info["ibm_marrakesh"]["TwoQubitGateErrors"].setdefault(gate_name, []).append(error)

cz
id
if_else
measure_2
rz
sx
x
xslow


In [67]:
# ----------------------------------
# SUMMARIZING MIN, MAX, MEAN, MEDIAN
# ----------------------------------

# Helper function to summarize properties of a backend
def summarize(values):
    np_array = np.array(values)
    return {
        "min": np.nanmin(np_array),
        "max": np.nanmax(np_array),
        "mean": np.nanmean(np_array),
        "median": np.nanmedian(np_array),
    }

# Summay for the Fez backend
fez_summary = {
    "T1": summarize(device_info["ibm_fez"]["T1"]),
    "T2": summarize(device_info["ibm_fez"]["T2"]),
    "Readout": summarize(device_info["ibm_fez"]["Readout"]),
}
for gate_name, errors in device_info["ibm_fez"]["SingleQubitGateErrors"].items():
    fez_summary[f"{gate_name.upper()} Error"] = summarize(errors)
for gate_name, errors in device_info["ibm_fez"]["TwoQubitGateErrors"].items():
    fez_summary[f"{gate_name.upper()} Error"] = summarize(errors)

# Summay for the Marrakesh backend
marrakesh_summary = {
    "T1": summarize(device_info["ibm_marrakesh"]["T1"]),
    "T2": summarize(device_info["ibm_marrakesh"]["T2"]),
    "Readout": summarize(device_info["ibm_marrakesh"]["Readout"]),
}
for gate_name, errors in device_info["ibm_marrakesh"]["SingleQubitGateErrors"].items():
    marrakesh_summary[f"{gate_name.upper()} Error"] = summarize(errors)
for gate_name, errors in device_info["ibm_marrakesh"]["TwoQubitGateErrors"].items():
    marrakesh_summary[f"{gate_name.upper()} Error"] = summarize(errors)

fez_summary_df = pd.DataFrame(fez_summary) 
marrakesh_summary_df = pd.DataFrame(marrakesh_summary)

display(fez_summary_df.style.set_caption("IBM FEZ SUMMARY"))
display(marrakesh_summary_df.style.set_caption("IBM MARRAKESH SUMMARY"))

,T1,T2,Readout,ID Error,MEASURE_2 Error,RZ Error,SX Error,X Error,XSLOW Error,CZ Error
min,17.761524,5.443194,0.292969,0.013850,0.195312,0.000000,0.013850,0.013850,0.013850,0.131041
max,331.350660,261.866226,33.703613,100.000000,42.553711,0.000000,100.000000,100.000000,100.000000,100.000000
mean,129.754101,95.989437,2.322544,0.683313,2.969752,0.000000,0.683313,0.683313,0.683313,3.386929
median,128.361346,90.712911,0.933838,0.029656,1.257324,0.000000,0.029656,0.029656,0.029656,0.284536


,T1,T2,Readout,ID Error,MEASURE_2 Error,RZ Error,SX Error,X Error,XSLOW Error,CZ Error
min,8.040057,6.034634,0.244141,0.012637,0.195312,0.000000,0.012637,0.012637,0.012637,0.100259
max,455.799366,433.491383,49.926758,100.000000,50.683594,0.000000,100.000000,100.000000,100.000000,100.000000
mean,165.914261,95.421604,4.351650,1.972796,4.073236,0.000000,1.972796,1.972796,1.972796,4.033829
median,154.296354,76.240297,1.348877,0.038560,1.452637,0.000000,0.038560,0.038560,0.038560,0.313319


### PART 2
In this part, I have:

- Chosen 2 available backends
- Read / store all properties from each qubit
- Find the min, max, median and mean for each property